In [ ]:
# A-Game-Theoretic-Equilibrium-Framework-for-Drift-Aware-Evolutionary-Clustering

import os
for _env_var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ.setdefault(_env_var, '1')
import sys
import json
import time
import shutil
import zipfile
import warnings
import importlib
import subprocess
import traceback
import random as _pyrandom
from collections import deque
warnings.filterwarnings('ignore')

def _ensure(pkg_import_name, pip_name=None):
    pip_name = pip_name or pkg_import_name
    try:
        return importlib.import_module(pkg_import_name)
    except ImportError:
        for extra_flags in ([], ['--break-system-packages']):
            try:
                subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name] + extra_flags)
                return importlib.import_module(pkg_import_name)
            except Exception:
                continue
        print(f"[setup] Could not install/import '{pip_name}' (no internet access or package unavailable) — continuing with a fallback.")
        return None
np = _ensure('numpy')
pd = _ensure('pandas')
plt = _ensure('matplotlib.pyplot', 'matplotlib')
_ensure('scipy')
_ensure('sklearn', 'scikit-learn')
river = _ensure('river')
from scipy.spatial import cKDTree
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
try:
    import resource

    def peak_memory_mb():
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0
except Exception:

    def peak_memory_mb():
        return float('nan')
RNG_SEED = 42
np.random.seed(RNG_SEED)
_pyrandom.seed(RNG_SEED)
OUT_DIR = '/content/gtec_outputs' if os.path.isdir('/content') else './gtec_outputs'
FIG_DIR = os.path.join(OUT_DIR, 'figures')
TAB_DIR = os.path.join(OUT_DIR, 'tables')
RAW_DIR = os.path.join(OUT_DIR, 'raw_results')
for d in (OUT_DIR, FIG_DIR, TAB_DIR, RAW_DIR):
    os.makedirs(d, exist_ok=True)
plt.rcParams.update({'figure.dpi': 130, 'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 10})
CONFIG = {'kdd': {'n_samples': 16000, 'window': 100}, 'pca': {'enabled': True, 'warmup_n': 1200, 'target_variance': 0.95, 'max_components': 15}, 'drift_log': {'chunk': 150, 'min_gap': 250}, 'gtec': {'max_micro_clusters': 20, 'n_candidates': 6, 'history_capacity': 10, 'kdtree_rebuild_every': 25, 'equilibrium_check_every': 50, 'drift_k_sigma': 3.0, 'novelty_k_sigma': 2.5, 'replicator_eta': 0.5, 'replicator_iters': 4, 'warmup_size': 250, 'initial_k_range': (2, 10), 'min_weight_disappear': 1.5, 'max_age_disappear': 1200, 'split_variance_factor': 2.2, 'merge_dist_factor': 0.6, 'history_blend_alpha': 0.65, 'history_match_sigma_mult': 3.0, 'confidence_weight_lo': 0.8, 'confidence_weight_span': 0.2, 'ot_sinkhorn_reg': 0.15, 'ot_sinkhorn_iters': 60, 'min_active_clusters': 2}, 'sensitivity': {'param': 'drift_k_sigma', 'values': (1.5, 3.0, 4.5)}, 'protocol': {'val_frac': 0.35, 'test_frac': 0.65}, 'covtype': {'enabled': True, 'n_samples': 6000, 'block_reorder': True}, 'seed': RNG_SEED}
print('=' * 80)
print('GTEC v4 Q1 ARTICLE EXPERIMENT PIPELINE — real KDD Cup 1999 + Forest Covertype streams')
print(f'river library available: {river is not None}')
print(f'Determinism: RNG_SEED={RNG_SEED}, BLAS threads pinned to 1')
try:
    import sklearn as _sk
    import scipy as _sp
    _ver_line = f'numpy={np.__version__} pandas={pd.__version__} sklearn={_sk.__version__} scipy={_sp.__version__}'
    if river is not None:
        _ver_line += f' river={river.__version__}'
    print(f'Library versions: {_ver_line}')
except Exception:
    pass
print('Output directory:', OUT_DIR)
print('=' * 80)
KDD_COARSE_MAP = {'normal.': 'normal', 'back.': 'dos', 'land.': 'dos', 'neptune.': 'dos', 'pod.': 'dos', 'smurf.': 'dos', 'teardrop.': 'dos', 'apache2.': 'dos', 'udpstorm.': 'dos', 'processtable.': 'dos', 'mailbomb.': 'dos', 'ipsweep.': 'probe', 'nmap.': 'probe', 'portsweep.': 'probe', 'satan.': 'probe', 'mscan.': 'probe', 'saint.': 'probe', 'ftp_write.': 'r2l', 'guess_passwd.': 'r2l', 'imap.': 'r2l', 'multihop.': 'r2l', 'phf.': 'r2l', 'spy.': 'r2l', 'warezclient.': 'r2l', 'warezmaster.': 'r2l', 'snmpgetattack.': 'r2l', 'snmpguess.': 'r2l', 'xlock.': 'r2l', 'xsnoop.': 'r2l', 'named.': 'r2l', 'sendmail.': 'r2l', 'worm.': 'r2l', 'buffer_overflow.': 'u2r', 'loadmodule.': 'u2r', 'perl.': 'u2r', 'rootkit.': 'u2r', 'httptunnel.': 'u2r', 'ps.': 'u2r', 'sqlattack.': 'u2r', 'xterm.': 'u2r'}

def load_kdd_stream(n_samples=10000, seed=42):
    try:
        from sklearn.datasets import fetch_kddcup99
        bunch = fetch_kddcup99(subset=None, percent10=True, as_frame=True, shuffle=False)
        df = bunch.frame.copy()
        label_col = 'labels'
        df[label_col] = df[label_col].apply(lambda b: b.decode() if isinstance(b, bytes) else str(b))
        df['coarse'] = df[label_col].map(lambda l: KDD_COARSE_MAP.get(l, 'other'))
        df = df[df['coarse'] != 'other']
        idx = np.linspace(0, len(df) - 1, num=min(n_samples, len(df))).astype(int)
        df = df.iloc[idx].reset_index(drop=True)
        cat_cols = [c for c in ['protocol_type', 'service', 'flag'] if c in df.columns]
        num_cols = [c for c in df.columns if c not in cat_cols + [label_col, 'coarse']]
        for c in cat_cols:
            df[c] = LabelEncoder().fit_transform(df[c].astype(str))
        for c in num_cols:
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0.0)
        feat_cols = cat_cols + num_cols
        X = df[feat_cols].values.astype(float)
        le = LabelEncoder().fit(df['coarse'].values)
        y = le.transform(df['coarse'].values)
        print(f'[data] KDD Cup 1999 (10%) loaded: {X.shape[0]} samples, {X.shape[1]} raw features, {len(le.classes_)} coarse attack classes ({', '.join(le.classes_)}).')
        return (X, y, 'KDD Cup 1999 (10%) — real-world network intrusion stream', list(le.classes_))
    except Exception as e:
        print(f"[data] Could not fetch KDD Cup 1999 ({e}); falling back to a synthetic multi-concept 'pseudo-real' stream (emergency offline fallback only — not a designed second dataset).")
        rng = np.random.RandomState(seed)
        n_clusters = 5
        d = 12
        centers = rng.uniform(-6, 6, size=(n_clusters, d))
        X, y = ([], [])
        for t in range(n_samples):
            phase = t / n_samples
            probs = np.abs(np.sin(np.linspace(0, 3.14, n_clusters) + phase * 6))
            probs /= probs.sum()
            lab = rng.choice(n_clusters, p=probs)
            X.append(centers[lab] + rng.normal(0, 1.0, size=d))
            y.append(lab)
        class_names = [f'class_{i}' for i in range(n_clusters)]
        return (np.array(X), np.array(y), 'Synthetic fallback (KDD Cup unavailable — no internet access)', class_names)

def reduce_dimensionality(X, warmup_n=1200, target_variance=0.95, max_components=15):
    warmup_n = min(warmup_n, len(X))
    n_comp_cap = max(2, min(max_components, X.shape[1], warmup_n - 1))
    probe = PCA(n_components=n_comp_cap, random_state=RNG_SEED).fit(X[:warmup_n])
    cum = np.cumsum(probe.explained_variance_ratio_)
    n_comp = int(np.searchsorted(cum, target_variance) + 1)
    n_comp = max(2, min(n_comp, n_comp_cap))
    pca = PCA(n_components=n_comp, random_state=RNG_SEED).fit(X[:warmup_n])
    X_reduced = pca.transform(X)
    variance_kept = float(cum[n_comp - 1]) if n_comp - 1 < len(cum) else float(cum[-1])
    print(f"[data] PCA representation: {X.shape[1]} -> {n_comp} dims ({100 * variance_kept:.1f}% variance retained, fit on the first {warmup_n} samples as a causal 'reference phase').")
    return (X_reduced, pca, n_comp, variance_kept)

def detect_label_shift_events(y, class_names, chunk=150, min_gap=250):
    n = len(y)
    events = []
    prev_dominant = None
    last_event_t = -10 ** 9
    for start in range(0, n, chunk):
        seg = y[start:start + chunk]
        if len(seg) == 0:
            continue
        vals, counts = np.unique(seg, return_counts=True)
        dominant = int(vals[np.argmax(counts)])
        if prev_dominant is not None and dominant != prev_dominant and (start - last_event_t >= min_gap):
            events.append({'t': start, 'type': f'shift_to_{class_names[dominant]}', 'detail': f'{class_names[prev_dominant]} -> {class_names[dominant]}'})
            last_event_t = start
        prev_dominant = dominant
    return events

def temporal_split(X, y, val_frac=0.35):
    n = len(X)
    n_val = int(round(n * val_frac))
    n_val = max(50, min(n_val, n - 50))
    return ((X[:n_val], y[:n_val]), (X[n_val:], y[n_val:]))

def load_covtype_stream(n_samples=6000, seed=42, block_reorder=True):
    try:
        from sklearn.datasets import fetch_covtype
        bunch = fetch_covtype()
        Xf = bunch.data.astype(float)
        yf = (bunch.target - 1).astype(int)
        classes = np.unique(yf)
        rng = np.random.RandomState(seed)
        per_class = max(20, n_samples // len(classes))
        blocks = []
        for cl in classes:
            cl_idx = np.where(yf == cl)[0]
            take = min(per_class, len(cl_idx))
            sel = rng.choice(cl_idx, size=take, replace=False)
            rng.shuffle(sel)
            blocks.append(sel)
        if block_reorder:
            order = rng.permutation(len(classes))
            seq = np.concatenate([blocks[i] for i in order])
        else:
            seq = rng.permutation(np.concatenate(blocks))
        Xseq = Xf[seq]
        prefix_n = max(50, int(0.2 * len(Xseq)))
        _cov_scaler = StandardScaler().fit(Xseq[:prefix_n])
        X = _cov_scaler.transform(Xseq)
        y = yf[seq]
        class_names = [f'cover_type_{c + 1}' for c in classes]
        print(f'[data] Forest Covertype loaded: {X.shape[0]} samples, {X.shape[1]} raw features, {len(classes)} classes{(' (class-block reordered into an evaluable drift stream)' if block_reorder else '')}.')
        return (X, y, 'Forest Covertype (UCI/sklearn) — second real dataset, class-block reordered', class_names)
    except Exception as e:
        print(f'[data] Could not fetch Forest Covertype ({e}); skipping the cross-dataset generalization check (Table 9 / Fig. 13 will be empty).')
        return (None, None, None, None)

class MicroCluster:
    __slots__ = ('id', 'N', 'LS', 'SS', 't_created', 't_last', 'hist_score', 't_last_reuse')

    def __init__(self, cid, x, t):
        self.id = cid
        self.N = 1.0
        self.LS = x.copy()
        self.SS = x * x
        self.t_created = t
        self.t_last = t
        self.hist_score = 0.0
        self.t_last_reuse = t

    def centroid(self):
        return self.LS / max(self.N, 1e-09)

    def variance(self):
        c = self.centroid()
        var = self.SS / max(self.N, 1e-09) - c * c
        return np.maximum(var, 1e-06)

    def radius(self):
        return float(np.sqrt(np.sum(self.variance())))

    def update(self, x, t, w=1.0, decay=0.995):
        self.N = self.N * decay + w
        self.LS = self.LS * decay + w * x
        self.SS = self.SS * decay + w * (x * x)
        self.t_last = t

    def signature(self):
        return np.concatenate([self.centroid(), [self.N, self.radius()]])

def sinkhorn_ot(a, b, cost, reg_frac=0.15, n_iter=60, eps=1e-09):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    scale = float(np.mean(cost)) + eps
    reg = max(reg_frac * scale, 1e-06)
    K = np.exp(-cost / reg)
    u = np.ones_like(a)
    v = np.ones_like(b)
    for _ in range(n_iter):
        Kv = K @ v
        u = a / np.maximum(Kv, eps)
        Ku = K.T @ u
        v = b / np.maximum(Ku, eps)
    plan = u[:, None] * K * v[None, :]
    distance = float(np.sum(plan * cost))
    return (plan, distance)

class HistoricalEquilibriumMemory:

    def __init__(self, capacity=10):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)

    def snapshot(self, clusters):
        if not clusters:
            return
        sig = np.array([c.signature() for c in clusters.values()])
        centroids = {cid: c.centroid().copy() for cid, c in clusters.items()}
        raw_w = np.array([c.N for c in clusters.values()], dtype=float)
        w_norm = raw_w / max(raw_w.sum(), 1e-09)
        weights = {cid: float(wi) for cid, wi in zip(clusters.keys(), w_norm)}
        self.buffer.append({'sig': sig, 'centroids': centroids, 'weights': weights})

    def best_match(self, clusters, threshold, use_ot=True, reg_frac=0.15, n_iter=60):
        if not self.buffer or not clusters:
            return None
        cur_ids = list(clusters.keys())
        cur_centroids = np.array([clusters[cid].centroid() for cid in cur_ids])
        raw_w = np.array([clusters[cid].N for cid in cur_ids], dtype=float)
        cur_weights = raw_w / max(raw_w.sum(), 1e-09)
        best_d, best_plan, best_hist_centroids = (np.inf, None, None)
        for entry in self.buffer:
            if entry['sig'].shape[0] == 0:
                continue
            hist_ids = list(entry['centroids'].keys())
            hist_centroids = np.array([entry['centroids'][hid] for hid in hist_ids])
            if use_ot:
                hist_w = np.array([entry['weights'].get(hid, 1.0) for hid in hist_ids], dtype=float)
                hist_w = hist_w / max(hist_w.sum(), 1e-09)
                cost = np.linalg.norm(cur_centroids[:, None, :] - hist_centroids[None, :, :], axis=2)
                plan, d = sinkhorn_ot(cur_weights, hist_w, cost, reg_frac=reg_frac, n_iter=n_iter)
            else:
                d = float(np.mean([np.min(np.linalg.norm(hist_centroids - c, axis=1)) for c in cur_centroids]))
                plan = None
            if d < best_d:
                best_d, best_plan, best_hist_centroids = (d, plan, hist_centroids)
        if best_hist_centroids is not None and best_d < threshold:
            return {'cur_ids': cur_ids, 'hist_centroids': best_hist_centroids, 'plan': best_plan, 'distance': best_d}
        return None

class GTEC:

    def __init__(self, n_dim, max_clusters=20, n_candidates=6, history_capacity=10, kdtree_rebuild_every=25, equilibrium_check_every=50, drift_k_sigma=3.0, novelty_k_sigma=2.5, replicator_eta=0.5, replicator_iters=4, warmup_size=250, initial_k_range=(2, 10), min_weight_disappear=1.5, max_age_disappear=1200, split_variance_factor=2.2, merge_dist_factor=0.6, history_blend_alpha=0.65, history_match_sigma_mult=3.0, confidence_weight_lo=0.8, confidence_weight_span=0.2, use_history=True, use_drift=True, use_game=True, adaptive_utility=True, use_ot_memory=True, ot_reg_frac=0.15, ot_n_iter=60, min_active_clusters=None, seed=42):
        self.n_dim = n_dim
        self.max_clusters = max_clusters
        self.r = n_candidates
        self.kdtree_rebuild_every = kdtree_rebuild_every
        self.equilibrium_check_every = equilibrium_check_every
        self.drift_k_sigma = drift_k_sigma
        self.novelty_k_sigma = novelty_k_sigma
        self.eta = replicator_eta
        self.replicator_iters = max(1, replicator_iters)
        self.warmup_size = warmup_size
        self.initial_k_range = initial_k_range
        self.min_weight_disappear = min_weight_disappear
        self.max_age_disappear = max_age_disappear
        self.split_variance_factor = split_variance_factor
        self.merge_dist_factor = merge_dist_factor
        self.history_blend_alpha = history_blend_alpha
        self.history_match_sigma_mult = history_match_sigma_mult
        self.confidence_weight_lo = confidence_weight_lo
        self.confidence_weight_span = confidence_weight_span
        self.ot_reg_frac = ot_reg_frac
        self.ot_n_iter = ot_n_iter
        self.min_active_clusters = min_active_clusters if min_active_clusters is not None else max(2, initial_k_range[0])
        self.use_history = use_history
        self.use_drift = use_drift
        self.use_game = use_game
        self.adaptive_utility = adaptive_utility
        self.use_ot_memory = use_ot_memory
        self.rng = np.random.RandomState(seed)
        self.clusters = {}
        self._next_id = 0
        self.memory = HistoricalEquilibriumMemory(history_capacity)
        self._tree = None
        self._tree_ids = []
        self._prev_equilibrium = None
        self._eq_mean = None
        self._eq_var = 1.0
        self._eq_alpha = 0.1
        self._dist_mean = None
        self._dist_var = 1.0
        self._dist_alpha = 0.05
        self._recent_buffer = deque(maxlen=400)
        self.t = 0
        self._initialized = False
        self._warmup_buffer = []
        self.initial_k_selected = None
        self.drift_events = []
        self.reuse_events = []
        self.split_merge_log = []
        self.last_ot_plan = None

    def _new_cluster(self, x, t):
        cid = self._next_id
        self._next_id += 1
        self.clusters[cid] = MicroCluster(cid, x, t)
        return cid

    def _rebuild_tree(self):
        if not self.clusters:
            self._tree, self._tree_ids = (None, [])
            return
        centroids = np.array([c.centroid() for c in self.clusters.values()])
        self._tree_ids = list(self.clusters.keys())
        self._tree = cKDTree(centroids)

    def _candidates(self, x):
        if self._tree is None or len(self._tree_ids) != len(self.clusters):
            self._rebuild_tree()
        if self._tree is None:
            return []
        k = min(self.r, len(self._tree_ids))
        _, idx = self._tree.query(x, k=k)
        idx = np.atleast_1d(idx)
        return [self._tree_ids[i] for i in idx if i < len(self._tree_ids)]

    def _update_dist_stats(self, d):
        if self._dist_mean is None:
            self._dist_mean, self._dist_var = (d, 1.0)
            return
        a = self._dist_alpha
        delta = d - self._dist_mean
        self._dist_mean += a * delta
        self._dist_var = (1 - a) * (self._dist_var + a * delta ** 2)

    def _novelty_threshold(self):
        if self._dist_mean is None:
            return 1000000000.0
        return self._dist_mean + self.novelty_k_sigma * np.sqrt(max(self._dist_var, 1e-06))

    def _memory_match_threshold(self):
        base = self._dist_mean if self._dist_mean is not None else 3.0
        return self.history_match_sigma_mult * base

    def _warm_start(self):
        Xw = np.array(self._warmup_buffer)
        if len(Xw) < 4:
            for x in Xw:
                self._new_cluster(x, self.t)
            self._rebuild_tree()
            return
        k_lo, k_hi = self.initial_k_range
        k_hi = min(k_hi, len(Xw) - 1, self.max_clusters)
        best_k, best_score, best_labels, best_centers = (None, -2.0, None, None)
        for k in range(max(2, k_lo), max(k_lo, k_hi) + 1):
            try:
                km = KMeans(n_clusters=k, n_init=5, random_state=RNG_SEED).fit(Xw)
                if len(np.unique(km.labels_)) < 2:
                    continue
                sub = min(len(Xw), 400)
                idx = np.random.RandomState(0).choice(len(Xw), size=sub, replace=False)
                score = silhouette_score(Xw[idx], km.labels_[idx])
                if score > best_score:
                    best_score, best_k = (score, k)
                    best_labels, best_centers = (km.labels_, km.cluster_centers_)
            except Exception:
                continue
        if best_k is None:
            best_k = min(4, max(2, k_hi))
            km = KMeans(n_clusters=best_k, n_init=5, random_state=RNG_SEED).fit(Xw)
            best_labels, best_centers = (km.labels_, km.cluster_centers_)
        for ci in range(best_k):
            pts = Xw[best_labels == ci]
            if len(pts) == 0:
                continue
            cid = self._next_id
            self._next_id += 1
            mc = MicroCluster(cid, pts[0], self.t)
            mc.N = float(len(pts))
            mc.LS = pts.sum(axis=0)
            mc.SS = (pts ** 2).sum(axis=0)
            mc.t_last = self.t
            self.clusters[cid] = mc
        self.initial_k_selected = best_k
        self._rebuild_tree()
        centroids = np.array([self.clusters[c].centroid() for c in self.clusters])
        for x in Xw[::max(1, len(Xw) // 50)]:
            d = float(np.min(np.linalg.norm(centroids - x, axis=1)))
            self._update_dist_stats(d)

    def _utility_weights(self):
        if not self.adaptive_utility:
            return dict(alpha=1.0, beta=1.0, gamma=1.0, delta=1.0, rho=0.5, lam=0.5, mu=0.2, nu=0.3)
        pressure = len(self.clusters) / max(self.max_clusters, 1)
        recent_drift_rate = len(self.drift_events[-5:]) / 5.0 if self.drift_events else 0.0
        return dict(alpha=1.2, beta=1.0 + 0.3 * pressure, gamma=1.0 - 0.3 * recent_drift_rate, delta=0.8 + 0.6 * recent_drift_rate, rho=0.4 + 0.4 * recent_drift_rate, lam=0.5 + 0.3 * pressure, mu=0.15 + 0.25 * pressure, nu=0.3)

    def _utility(self, x, cand_ids, w):
        U = {}
        for cid in cand_ids:
            c = self.clusters[cid]
            centroid = c.centroid()
            r = c.radius() + 0.001
            dist = np.linalg.norm(x - centroid) + 1e-09
            S = np.exp(-dist ** 2 / (2 * r ** 2))
            D = S * np.tanh(c.N / (1.0 + r) / 10.0)
            Stab = 1.0 / (1.0 + (self.t - c.t_last) / 50.0)
            Hist = c.hist_score * np.exp(-(self.t - c.t_last_reuse) / 300.0)
            Var = np.tanh(np.mean(c.variance()))
            Mem = len(self.clusters) / max(self.max_clusters, 1)
            AgeRisk = np.tanh((self.t - c.t_created) / 500.0)
            U[cid] = w['alpha'] * S + w['beta'] * D + w['gamma'] * Stab + w['delta'] * Hist - w['lam'] * Var - w['mu'] * Mem - w['nu'] * AgeRisk
        return U

    def _birth_utility(self, best_dist, cand_ids, w):
        raw_scale = float(np.mean([self.clusters[cid].radius() for cid in cand_ids])) + 0.001
        if self._dist_mean is not None:
            cap = self.history_match_sigma_mult * (self._dist_mean + 3.0 * np.sqrt(max(self._dist_var, 1e-06)))
            scale = min(raw_scale, max(cap, 0.001))
        else:
            scale = raw_scale
        S_best = np.exp(-best_dist ** 2 / (2 * scale ** 2))
        mem_if_added = (len(self.clusters) + 1) / max(self.max_clusters, 1)
        return w['rho'] * (1.0 - S_best) - w['mu'] * mem_if_added

    def _equilibrium_summary(self):
        if not self.clusters:
            return np.zeros(4)
        weights = np.array([c.N for c in self.clusters.values()])
        radii = np.array([c.radius() for c in self.clusters.values()])
        w_frac = weights / weights.sum()
        entropy = -np.sum(w_frac * np.log(w_frac + 1e-09))
        return np.array([len(self.clusters), float(np.mean(radii)), float(entropy), float(np.mean(weights))])

    def _check_drift_equilibrium(self):
        E_t = self._equilibrium_summary()
        drifted = False
        if self._prev_equilibrium is not None:
            D_E = float(np.linalg.norm(E_t - self._prev_equilibrium))
            if self._eq_mean is None:
                self._eq_mean, self._eq_var = (D_E, 1.0)
            threshold = self._eq_mean + self.drift_k_sigma * np.sqrt(self._eq_var)
            if self.use_drift and D_E > threshold and (self.t > self.warmup_size + 150):
                drifted = True
                self.drift_events.append(self.t)
            a = self._eq_alpha
            delta = D_E - self._eq_mean
            self._eq_mean += a * delta
            self._eq_var = (1 - a) * (self._eq_var + a * delta ** 2)
        self._prev_equilibrium = E_t
        return (drifted, E_t)

    def _structural_adaptation(self):
        ids = list(self.clusters.keys())
        for cid in ids:
            if len(self.clusters) <= self.min_active_clusters:
                break
            c = self.clusters[cid]
            if c.N < self.min_weight_disappear and self.t - c.t_last > self.max_age_disappear:
                del self.clusters[cid]
                self.split_merge_log.append({'t': self.t, 'event': 'disappear', 'cluster': cid})
        ids = list(self.clusters.keys())
        centroids = {cid: self.clusters[cid].centroid() for cid in ids}
        merged = set()
        for i, a in enumerate(ids):
            if len(self.clusters) - len(merged) <= self.min_active_clusters:
                break
            if a in merged or a not in self.clusters:
                continue
            for b in ids[i + 1:]:
                if len(self.clusters) - len(merged) <= self.min_active_clusters:
                    break
                if b in merged or b not in self.clusters:
                    continue
                dist = np.linalg.norm(centroids[a] - centroids[b])
                scale = 0.5 * (self.clusters[a].radius() + self.clusters[b].radius() + 0.001)
                if dist < self.merge_dist_factor * scale:
                    ca, cb = (self.clusters[a], self.clusters[b])
                    ca.N += cb.N
                    ca.LS += cb.LS
                    ca.SS += cb.SS
                    del self.clusters[b]
                    merged.add(b)
                    self.split_merge_log.append({'t': self.t, 'event': 'merge', 'clusters': (a, b)})
        ids = list(self.clusters.keys())
        radii = [self.clusters[cid].radius() for cid in ids] if ids else [0]
        mean_r = np.mean(radii) if radii else 1.0
        recent_X = np.array([x for x, t in self._recent_buffer]) if self._recent_buffer else np.zeros((0, self.n_dim))
        for cid in ids:
            if len(self.clusters) >= self.max_clusters:
                break
            c = self.clusters[cid]
            if c.radius() > self.split_variance_factor * (mean_r + 1e-06) and c.N > 4 and (len(recent_X) >= 6):
                dists = np.linalg.norm(recent_X - c.centroid(), axis=1)
                local = recent_X[dists < 2 * c.radius() + 0.001]
                if len(local) >= 4:
                    try:
                        km = KMeans(n_clusters=2, n_init=3, random_state=RNG_SEED).fit(local)
                        c0, c1 = km.cluster_centers_
                        new_id = self._new_cluster(c1, self.t)
                        c.LS = c0 * (c.N / 2.0)
                        c.SS = c0 * c0 * (c.N / 2.0)
                        self.clusters[new_id].N = c.N / 2.0
                        self.clusters[new_id].LS = c1 * (c.N / 2.0)
                        self.clusters[new_id].SS = c1 * c1 * (c.N / 2.0)
                        c.N = c.N / 2.0
                        self.split_merge_log.append({'t': self.t, 'event': 'split', 'cluster': cid, 'new': new_id})
                    except Exception:
                        pass

    def _try_historical_reuse(self):
        thr = self._memory_match_threshold()
        match = self.memory.best_match(self.clusters, threshold=thr, use_ot=self.use_ot_memory, reg_frac=self.ot_reg_frac, n_iter=self.ot_n_iter)
        if match is None:
            return
        self.reuse_events.append(self.t)
        hist_centroids = match['hist_centroids']
        alpha = self.history_blend_alpha
        if match['plan'] is not None:
            plan = match['plan']
            m_hist = hist_centroids.shape[0]
            for i, cid in enumerate(match['cur_ids']):
                if cid not in self.clusters:
                    continue
                c = self.clusters[cid]
                row = plan[i]
                row_sum = float(row.sum())
                if row_sum <= 1e-09:
                    continue
                target = row @ hist_centroids / row_sum
                peak_share = row.max() / row_sum
                conf = float(np.clip((peak_share - 1.0 / m_hist) / max(1e-06, 1 - 1.0 / m_hist), 0.0, 1.0)) if m_hist > 1 else 1.0
                eff_alpha = alpha * conf
                if eff_alpha < 0.001:
                    continue
                blended = (1 - eff_alpha) * c.centroid() + eff_alpha * target
                c.LS = blended * c.N
                c.hist_score = min(1.0, c.hist_score + 0.3 * conf)
                c.t_last_reuse = self.t
            self.last_ot_plan = {'plan': plan, 'cur_ids': match['cur_ids'], 'hist_centroids': hist_centroids, 't': self.t}
        else:
            for cid, c in self.clusters.items():
                d = np.linalg.norm(hist_centroids - c.centroid(), axis=1)
                j = int(np.argmin(d))
                if d[j] < thr:
                    conf = float(np.clip(1.0 - d[j] / thr, 0.0, 1.0))
                    eff_alpha = alpha * conf
                    if eff_alpha < 0.001:
                        continue
                    blended = (1 - eff_alpha) * c.centroid() + eff_alpha * hist_centroids[j]
                    c.LS = blended * c.N
                    c.hist_score = min(1.0, c.hist_score + 0.3 * conf)
                    c.t_last_reuse = self.t

    def learn_one(self, x, t=None):
        self.t = t if t is not None else self.t + 1
        self._recent_buffer.append((x.copy(), self.t))
        if not self._initialized:
            self._warmup_buffer.append(x.copy())
            if len(self._warmup_buffer) >= self.warmup_size:
                self._warm_start()
                self._initialized = True
            return
        if len(self.clusters) == 0:
            self._new_cluster(x, self.t)
            return
        if len(self.clusters) < self.min_active_clusters and len(self.clusters) < self.max_clusters:
            self._new_cluster(x, self.t)
            return
        cand_ids = self._candidates(x)
        if not cand_ids:
            if len(self.clusters) < self.max_clusters:
                self._new_cluster(x, self.t)
            return
        best_cid0 = min(cand_ids, key=lambda cid: np.linalg.norm(x - self.clusters[cid].centroid()))
        best_dist = float(np.linalg.norm(x - self.clusters[best_cid0].centroid()))
        self._update_dist_stats(best_dist)
        w = self._utility_weights()
        if self.use_game:
            U = self._utility(x, cand_ids, w)
            U_birth = self._birth_utility(best_dist, cand_ids, w)
            u = np.array([U[c] for c in cand_ids] + [U_birth])
            p = np.ones(len(u)) / len(u)
            for _ in range(self.replicator_iters):
                u_bar = float(np.sum(p * u))
                p = p + self.eta * p * (u - u_bar)
                p = np.clip(p, 1e-06, None)
                p /= p.sum()
            winner = int(np.argmax(p))
            is_birth = winner == len(cand_ids)
            if is_birth and len(self.clusters) < self.max_clusters:
                self._new_cluster(x, self.t)
            else:
                if is_birth:
                    winner = int(np.argmax(p[:-1]))
                best_cid = cand_ids[winner]
                confidence = float(p[winner])
                update_w = self.confidence_weight_lo + self.confidence_weight_span * confidence
                self.clusters[best_cid].update(x, self.t, w=update_w)
        elif best_dist > self._novelty_threshold() and len(self.clusters) < self.max_clusters:
            self._new_cluster(x, self.t)
        else:
            U = self._utility(x, cand_ids, w)
            best_idx = int(np.argmax(list(U.values())))
            best_cid = cand_ids[best_idx]
            self.clusters[best_cid].update(x, self.t)
        if len(self.clusters) != len(self._tree_ids):
            self._rebuild_tree()
        elif self.t % self.kdtree_rebuild_every == 0:
            self._rebuild_tree()
        if self.t % self.equilibrium_check_every == 0:
            drifted, _ = self._check_drift_equilibrium()
            if drifted:
                self._structural_adaptation()
                if self.use_history:
                    self._try_historical_reuse()
                self._rebuild_tree()
            elif self.use_history:
                self.memory.snapshot(self.clusters)

    def predict_one(self, x):
        if not self.clusters:
            return -1
        centroids = np.array([c.centroid() for c in self.clusters.values()])
        ids = list(self.clusters.keys())
        j = int(np.argmin(np.linalg.norm(centroids - x, axis=1)))
        return ids[j]

class _RiverAdapter:

    def __init__(self, model):
        self.model = model
        self._warned = False

    @staticmethod
    def _to_dict(x):
        return {i: float(v) for i, v in enumerate(x)}

    def learn_one(self, x, t=None):
        d = self._to_dict(x)
        self.model.learn_one(d)

    def _center_vectors(self, dim):
        for attr in ('p_micro_clusters', 'clusters', 'centers', 'micro_clusters', '_clusters', 'centroids'):
            val = getattr(self.model, attr, None)
            if not val:
                continue
            try:
                items = list(val.items()) if hasattr(val, 'items') else list(enumerate(val))
                ids, vecs = ([], [])
                for i, obj in items:
                    c = getattr(obj, 'center', obj)
                    v = np.array([c.get(j, 0.0) for j in range(dim)]) if isinstance(c, dict) else np.asarray(c, dtype=float)
                    if v.shape[0] == dim:
                        ids.append(i)
                        vecs.append(v)
                if vecs:
                    return (ids, np.array(vecs))
            except Exception:
                continue
        return (None, None)

    def predict_one(self, x):
        d = self._to_dict(x)
        raised = False
        try:
            out = int(self.model.predict_one(d))
        except Exception as e:
            raised = True
            out = -1
            if not self._warned:
                print(f'[baseline fallback] {type(self.model).__name__} predict_one raised: {e}')
                self._warned = True
        if out < 0:
            ids, vecs = self._center_vectors(len(x))
            if vecs is not None and len(vecs):
                j = int(np.argmin(np.linalg.norm(vecs - x, axis=1)))
                return int(ids[j]) if isinstance(ids[j], (int, np.integer)) else j
            if not raised and (not self._warned):
                print(f'[baseline fallback] {type(self.model).__name__} predict_one returned {out}, no centers exposed yet')
                self._warned = True
            return -1
        return out

    @property
    def clusters(self):
        for attr in ('p_micro_clusters', 'clusters', 'centers', 'micro_clusters', '_clusters', 'centroids'):
            val = getattr(self.model, attr, None)
            if val is not None:
                try:
                    return {i: None for i in range(len(val))}
                except TypeError:
                    continue
        return {}

class _LiteMicroClusterBaseline:

    def __init__(self, mode, max_clusters=20, radius_factor=1.5, seed=42, n_target_clusters=5):
        self.mode = mode
        self.max_clusters = max_clusters
        self.radius_factor = radius_factor
        self.rng = np.random.RandomState(seed)
        self.clusters = {}
        self._next_id = 0
        self.t = 0
        self.lr = 0.05
        self.n_target_clusters = n_target_clusters

    def _new(self, x):
        cid = self._next_id
        self._next_id += 1
        self.clusters[cid] = MicroCluster(cid, x, self.t)
        return cid

    def learn_one(self, x, t=None):
        self.t = t if t is not None else self.t + 1
        if not self.clusters:
            self._new(x)
            return
        centroids = np.array([c.centroid() for c in self.clusters.values()])
        ids = list(self.clusters.keys())
        d = np.linalg.norm(centroids - x, axis=1)
        j = int(np.argmin(d))
        cid = ids[j]
        c = self.clusters[cid]
        r = c.radius() + 0.001
        if self.mode == 'streamkm':
            if len(self.clusters) < self.n_target_clusters and d[j] > self.radius_factor * r:
                self._new(x)
                return
            c.LS = c.centroid() * (1 - self.lr) * c.N + self.lr * x * c.N
            c.N += 1
            return
        if d[j] <= self.radius_factor * r or len(self.clusters) >= self.max_clusters:
            c.update(x, self.t, decay=0.999 if self.mode == 'denstream' else 1.0)
            if self.mode == 'dbstream' and self.t % 40 == 0:
                self._merge_overlapping()
        else:
            self._new(x)
            if self.mode == 'denstream' and self.t % 60 == 0:
                self._prune_outliers()

    def _merge_overlapping(self):
        ids = list(self.clusters.keys())
        for i, a in enumerate(ids):
            if a not in self.clusters:
                continue
            for b in ids[i + 1:]:
                if b not in self.clusters:
                    continue
                ca, cb = (self.clusters[a], self.clusters[b])
                dist = np.linalg.norm(ca.centroid() - cb.centroid())
                if dist < 0.5 * (ca.radius() + cb.radius() + 0.001):
                    ca.N += cb.N
                    ca.LS += cb.LS
                    ca.SS += cb.SS
                    del self.clusters[b]

    def _prune_outliers(self):
        for cid in list(self.clusters.keys()):
            if self.clusters[cid].N < 1.2 and self.t - self.clusters[cid].t_last > 300:
                if len(self.clusters) > 2:
                    del self.clusters[cid]

    def predict_one(self, x):
        if not self.clusters:
            return -1
        centroids = np.array([c.centroid() for c in self.clusters.values()])
        ids = list(self.clusters.keys())
        return ids[int(np.argmin(np.linalg.norm(centroids - x, axis=1)))]

def calibrate_river_baseline(ctor, candidates, X_val, y_val, window, min_healthy_frac=0.5):
    best_kw, best_frac = (candidates[0], -1.0)
    for kw in candidates:
        try:
            model = _RiverAdapter(ctor(**kw))
            df, _ = run_stream_experiment('calib', model, X_val, y_val, window=window, verbose=False)
            frac = float(df['ARI'].notna().mean()) if len(df) else 0.0
        except Exception:
            frac = 0.0
        if frac > best_frac:
            best_kw, best_frac = (kw, frac)
        if frac >= min_healthy_frac:
            return (kw, frac)
    return (best_kw, best_frac)

def build_baselines(n_true_clusters, denstream_kwargs=None, dbstream_kwargs=None):
    baselines = {}
    denstream_kwargs = denstream_kwargs or dict(decaying_factor=0.01, beta=0.5, mu=1.2)
    dbstream_kwargs = dbstream_kwargs or dict(clustering_threshold=1.5, fading_factor=0.01)
    if river is not None:
        try:
            from river import cluster as rc
            baselines['CluStream'] = _RiverAdapter(rc.CluStream(n_macro_clusters=n_true_clusters, max_micro_clusters=30, time_window=1000, seed=RNG_SEED))
        except Exception as e:
            print('[baseline] river CluStream unavailable, using fallback:', e)
        try:
            from river import cluster as rc
            baselines['DenStream'] = _RiverAdapter(rc.DenStream(**denstream_kwargs))
        except Exception as e:
            print('[baseline] river DenStream unavailable, using fallback:', e)
        try:
            from river import cluster as rc
            baselines['DBSTREAM'] = _RiverAdapter(rc.DBSTREAM(**dbstream_kwargs))
        except Exception as e:
            print('[baseline] river DBSTREAM unavailable, using fallback:', e)
        try:
            from river import cluster as rc
            baselines['StreamKM++'] = _RiverAdapter(rc.STREAMKMeans(chunk_size=100, n_clusters=n_true_clusters, seed=RNG_SEED))
        except Exception as e:
            print('[baseline] river STREAMKMeans unavailable, using fallback:', e)
    for name, mode in [('CluStream', 'clustream'), ('DenStream', 'denstream'), ('DBSTREAM', 'dbstream'), ('StreamKM++', 'streamkm')]:
        if name not in baselines:
            baselines[name] = _LiteMicroClusterBaseline(mode=mode, max_clusters=max(15, n_true_clusters * 3), n_target_clusters=max(2, n_true_clusters))
    return baselines

def purity_score(y_true, y_pred):
    y_true, y_pred = (np.asarray(y_true), np.asarray(y_pred))
    total = 0
    for cl in np.unique(y_pred):
        mask = y_pred == cl
        if mask.sum() == 0:
            continue
        vals, counts = np.unique(y_true[mask], return_counts=True)
        total += counts.max()
    return total / len(y_true)

def run_stream_experiment(name, model, X, y_true, window=300, drift_log=None, verbose=True):
    n = len(X)
    rows = []
    buf_X, buf_y = ([], [])
    t0_all = time.perf_counter()
    per_sample_times = []
    for t in range(n):
        x = X[t]
        t1 = time.perf_counter()
        model.learn_one(x, t)
        per_sample_times.append(time.perf_counter() - t1)
        buf_X.append(x)
        buf_y.append(y_true[t])
        if (t + 1) % window == 0 or t == n - 1:
            wX = np.array(buf_X)
            wY = np.array(buf_y)
            preds = np.array([model.predict_one(xx) for xx in wX])
            valid = preds >= 0
            if valid.sum() >= 2:
                try:
                    ari = adjusted_rand_score(wY[valid], preds[valid])
                    nmi = normalized_mutual_info_score(wY[valid], preds[valid])
                    pur = purity_score(wY[valid], preds[valid])
                except Exception:
                    ari = nmi = pur = np.nan
                try:
                    sub = min(300, valid.sum())
                    sel = np.random.RandomState(0).choice(np.where(valid)[0], size=sub, replace=False)
                    sil = silhouette_score(wX[sel], preds[sel]) if len(np.unique(preds[sel])) >= 2 else np.nan
                except Exception:
                    sil = np.nan
            else:
                ari = nmi = pur = sil = np.nan
            rows.append({'method': name, 't': t, 'ARI': ari, 'NMI': nmi, 'Purity': pur, 'Silhouette': sil, 'n_clusters': len(getattr(model, 'clusters', {}) or {})})
            buf_X, buf_y = ([], [])
    total_time = time.perf_counter() - t0_all
    mem = peak_memory_mb()
    df = pd.DataFrame(rows)
    throughput = n / max(total_time, 1e-09)
    avg_ms = 1000 * np.mean(per_sample_times)
    if verbose:
        final_ari = df['ARI'].dropna().iloc[-1] if df['ARI'].notna().any() else float('nan')
        print(f'  [{name:14s}] {total_time:6.2f}s | throughput={throughput:8.1f}/s | avg={avg_ms:.3f} ms/sample | mem={mem:.1f} MB | mean ARI={df['ARI'].mean():.3f} | final ARI={final_ari:.3f}')
    extra = {'total_time_s': total_time, 'throughput_sps': throughput, 'avg_ms_per_sample': avg_ms, 'peak_memory_mb': mem}
    if drift_log:
        drift_ts = [e['t'] for e in drift_log]
        drift_types = [e['type'] for e in drift_log]
        recov_rows = []
        detected = getattr(model, 'drift_events', None)
        for dt, dtype in zip(drift_ts, drift_types):
            pre = df[df['t'] < dt]['ARI'].dropna()
            post = df[df['t'] >= dt]
            baseline_ari = pre.iloc[-3:].mean() if len(pre) >= 1 else np.nan
            recovery_t = np.nan
            if not np.isnan(baseline_ari):
                target = 0.9 * baseline_ari
                ok_mask = (post['ARI'] >= target).values
                for i in range(len(ok_mask) - 1):
                    if ok_mask[i] and ok_mask[i + 1]:
                        recovery_t = int(post['t'].values[i] - dt)
                        break
            delay = np.nan
            if detected:
                after = [d for d in detected if d >= dt]
                if after:
                    delay = after[0] - dt
            recov_rows.append({'method': name, 'drift_t': dt, 'drift_type': dtype, 'detection_delay': delay, 'recovery_time_samples': recovery_t})
        extra['recovery_df'] = pd.DataFrame(recov_rows)
    return (df, extra)

def make_gtec_factory(n_dim):
    gcfg = CONFIG['gtec']

    def make(**overrides):
        kw = dict(n_dim=n_dim, max_clusters=gcfg['max_micro_clusters'], n_candidates=gcfg['n_candidates'], history_capacity=gcfg['history_capacity'], kdtree_rebuild_every=gcfg['kdtree_rebuild_every'], equilibrium_check_every=gcfg['equilibrium_check_every'], drift_k_sigma=gcfg['drift_k_sigma'], novelty_k_sigma=gcfg['novelty_k_sigma'], replicator_eta=gcfg['replicator_eta'], replicator_iters=gcfg['replicator_iters'], warmup_size=gcfg['warmup_size'], initial_k_range=gcfg['initial_k_range'], min_weight_disappear=gcfg['min_weight_disappear'], max_age_disappear=gcfg['max_age_disappear'], split_variance_factor=gcfg['split_variance_factor'], merge_dist_factor=gcfg['merge_dist_factor'], history_blend_alpha=gcfg['history_blend_alpha'], history_match_sigma_mult=gcfg['history_match_sigma_mult'], confidence_weight_lo=gcfg['confidence_weight_lo'], confidence_weight_span=gcfg['confidence_weight_span'], ot_reg_frac=gcfg['ot_sinkhorn_reg'], ot_n_iter=gcfg['ot_sinkhorn_iters'], min_active_clusters=gcfg['min_active_clusters'], seed=RNG_SEED)
        kw.update(overrides)
        return GTEC(**kw)
    return make

def build_gtec_variants(n_dim):
    make = make_gtec_factory(n_dim)
    return {'GTEC (full)': make(), 'GTEC w/o History': make(use_history=False), 'GTEC w/o OT Memory (v3 nearest-centroid)': make(use_ot_memory=False), 'GTEC w/o Drift-Trigger': make(use_drift=False), 'GTEC w/o Game (hard assign)': make(use_game=False), 'GTEC w/o Adaptive Utility': make(adaptive_utility=False)}

def run_sensitivity_sweep(n_dim, X, y, window, param_name, values):
    make = make_gtec_factory(n_dim)
    rows = []
    for v in values:
        model = make(**{param_name: v})
        df, _ = run_stream_experiment(f'{param_name}={v}', model, X, y, window=window, drift_log=None, verbose=False)
        rows.append({'param': param_name, 'value': v, 'ARI_mean': df['ARI'].mean(), 'NMI_mean': df['NMI'].mean()})
        print(f'  [sensitivity] {param_name}={v}: mean ARI={df['ARI'].mean():.3f}, mean NMI={df['NMI'].mean():.3f}')
    return pd.DataFrame(rows)

def build_complexity_table():
    rows = [{'method': 'GTEC (proposed)', 'per_point_time': 'O(r log k + r)', 'memory': 'O(k*d + M*k*d)', 'notes': 'r=candidates via KD-tree (r<<k), k=active clusters, d=dims, M=history capacity'}, {'method': 'CluStream', 'per_point_time': 'O(k)', 'memory': 'O(k*d)', 'notes': 'linear scan over k micro-clusters'}, {'method': 'DenStream', 'per_point_time': 'O(k)', 'memory': 'O(k*d)', 'notes': 'potential + outlier micro-cluster lists'}, {'method': 'DBSTREAM', 'per_point_time': 'O(k)', 'memory': 'O(k*d + k^2)', 'notes': 'pairwise shared-density tracking'}, {'method': 'StreamKM++', 'per_point_time': 'O(k)', 'memory': 'O(k*d)', 'notes': 'online seeding + sequential k-means update'}]
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(TAB_DIR, 'Table5_complexity_comparison.csv'), index=False)
    return df
_NEMENYI_Q005 = {2: 1.96, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.85, 7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164, 11: 3.219, 12: 3.268}

def critical_difference(k, n_blocks, alpha=0.05):
    if alpha != 0.05:
        raise ValueError('only the alpha=0.05 table is tabulated here')
    q = _NEMENYI_Q005.get(k, _NEMENYI_Q005[min(_NEMENYI_Q005, key=lambda kk: abs(kk - k))])
    return q * np.sqrt(k * (k + 1) / (6.0 * n_blocks))

def build_significance_tables(dfs, reference='GTEC (full)', metric='ARI'):
    if reference not in dfs or reference not in dfs.keys():
        return (pd.DataFrame(), pd.DataFrame())
    ref_vals = dfs[reference][metric].reset_index(drop=True)
    others = [m for m in dfs if m != reference]
    raw_p, rows = ([], [])
    for m in others:
        v = dfs[m][metric].reset_index(drop=True)
        n = min(len(ref_vals), len(v))
        a, b = (ref_vals.iloc[:n], v.iloc[:n])
        mask = a.notna() & b.notna() & ((a - b).abs() > 1e-12)
        a, b = (a[mask], b[mask])
        if len(a) < 6:
            rows.append({'comparison': f'{reference} vs {m}', 'metric': metric, 'n_windows': int(len(a)), 'median_diff': np.nan, 'wilcoxon_stat': np.nan, 'p_raw': np.nan})
            raw_p.append(np.nan)
            continue
        stat, p = stats.wilcoxon(a, b)
        rows.append({'comparison': f'{reference} vs {m}', 'metric': metric, 'n_windows': int(len(a)), 'median_diff': float(np.median(a - b)), 'wilcoxon_stat': float(stat), 'p_raw': float(p)})
        raw_p.append(p)
    valid_idx = [i for i, p in enumerate(raw_p) if not np.isnan(p)]
    order = sorted(valid_idx, key=lambda i: raw_p[i])
    m_tests = len(order)
    p_holm = {i: np.nan for i in range(len(raw_p))}
    running_max = 0.0
    for rank, i in enumerate(order):
        adj = (m_tests - rank) * raw_p[i]
        running_max = max(running_max, adj)
        p_holm[i] = float(min(1.0, running_max))
    for i, row in enumerate(rows):
        row['p_holm'] = p_holm[i]
        row['significant_at_0.05'] = bool(p_holm[i] < 0.05) if not np.isnan(p_holm[i]) else False
    table7 = pd.DataFrame(rows)
    all_methods = list(dfs.keys())
    n_rows = min((len(dfs[m]) for m in all_methods))
    coverage = {m: float(dfs[m][metric].notna().values[:n_rows].mean()) for m in all_methods}
    min_blocks = 15
    order = sorted(all_methods, key=lambda m: coverage[m], reverse=True)
    included = []
    running_valid = np.ones(n_rows, dtype=bool)
    for m in order:
        candidate_valid = running_valid & dfs[m][metric].notna().values[:n_rows]
        if candidate_valid.sum() >= min_blocks or not included:
            running_valid = candidate_valid
            included.append(m)
    excluded = [m for m in all_methods if m not in included]
    if excluded:
        print(f'[Table 8] excluded (insufficient joint valid-window overlap): {excluded}')
    aligned = pd.concat([dfs[m][metric].reset_index(drop=True).rename(m) for m in included], axis=1)
    aligned = aligned.dropna()
    fried_rows = []
    if len(aligned) >= 6 and aligned.shape[1] >= 3:
        stat_f, p_f = stats.friedmanchisquare(*[aligned[m].values for m in included])
        ranks = aligned.rank(axis=1, ascending=False)
        avg_rank = ranks.mean(axis=0).sort_values()
        cd = critical_difference(k=len(included), n_blocks=len(aligned))
        for m in avg_rank.index:
            fried_rows.append({'method': m, 'avg_rank': float(avg_rank[m]), 'n_blocks': int(len(aligned)), 'friedman_chi2': float(stat_f), 'friedman_p': float(p_f), 'critical_difference': float(cd)})
    table8 = pd.DataFrame(fried_rows)
    return (table7, table8)

def run_all():
    results = {}
    print('\n[1/6] Loading the real-world KDD Cup 1999 (10%) stream ...')
    X_raw, y, kdd_desc, class_names = load_kdd_stream(CONFIG['kdd']['n_samples'], seed=RNG_SEED)
    proto = CONFIG['protocol']
    (Xraw_val, y_val), (Xraw_test, y_test) = temporal_split(X_raw, y, val_frac=proto['val_frac'])
    print(f'[protocol] {len(Xraw_val)} samples -> VALIDATION (tuning only), {len(Xraw_test)} samples -> TEST (touched once, for every reported number below).')
    _feat_scaler = StandardScaler().fit(Xraw_val)
    Xraw_val = _feat_scaler.transform(Xraw_val)
    Xraw_test = _feat_scaler.transform(Xraw_test)
    if CONFIG['pca']['enabled']:
        X_val, pca_model, n_comp, var_kept = reduce_dimensionality(Xraw_val, warmup_n=CONFIG['pca']['warmup_n'], target_variance=CONFIG['pca']['target_variance'], max_components=CONFIG['pca']['max_components'])
        X_test = pca_model.transform(Xraw_test)
    else:
        X_val, X_test, n_comp, var_kept = (Xraw_val, Xraw_test, X_raw.shape[1], 1.0)
    n_true = len(np.unique(y_val))
    drift_log_test = detect_label_shift_events(y_test, class_names, chunk=CONFIG['drift_log']['chunk'], min_gap=CONFIG['drift_log']['min_gap'])
    print(f"[data] {len(drift_log_test)} real concept-drift (label-shift) events detected in the TEST segment's ground-truth sequence (used only for evaluation).")
    dataset_summary = pd.DataFrame([{'dataset': kdd_desc, 'n_samples_total': X_raw.shape[0], 'n_samples_val': X_val.shape[0], 'n_samples_test': X_test.shape[0], 'n_raw_features': X_raw.shape[1], 'n_reduced_features': n_comp, 'pca_variance_retained': round(var_kept, 4), 'n_true_clusters': n_true, 'true_classes': ', '.join(class_names), 'n_ground_truth_drift_events_test': len(drift_log_test)}])
    dataset_summary.to_csv(os.path.join(TAB_DIR, 'Table1_dataset_summary.csv'), index=False)
    print('\n[2/6] Hyperparameter sensitivity sweep — VALIDATION segment only ...')
    sens_cfg = CONFIG['sensitivity']
    sensitivity_df = run_sensitivity_sweep(X_val.shape[1], X_val, y_val, CONFIG['kdd']['window'], sens_cfg['param'], sens_cfg['values'])
    sensitivity_df.to_csv(os.path.join(TAB_DIR, 'Table6_sensitivity_sweep_VALIDATION.csv'), index=False)
    print('[3/6] Running proposed method (GTEC) + ablations on the TEST segment ...')
    variants = build_gtec_variants(n_dim=X_test.shape[1])
    print('[4/6] Running baselines (CluStream, DenStream, DBSTREAM, StreamKM++) on the TEST segment ...')
    denstream_kw, dbstream_kw = (None, None)
    if river is not None:
        try:
            from river import cluster as rc
            denstream_candidates = [dict(decaying_factor=0.01, beta=0.5, mu=1.2), dict(decaying_factor=0.02, beta=0.4, mu=1.0), dict(decaying_factor=0.05, beta=0.3, mu=0.5)]
            denstream_kw, _ = calibrate_river_baseline(rc.DenStream, denstream_candidates, X_val, y_val, CONFIG['kdd']['window'])
            dbstream_candidates = [dict(clustering_threshold=1.5, fading_factor=0.01), dict(clustering_threshold=1.0, fading_factor=0.02), dict(clustering_threshold=0.7, fading_factor=0.05)]
            dbstream_kw, _ = calibrate_river_baseline(rc.DBSTREAM, dbstream_candidates, X_val, y_val, CONFIG['kdd']['window'])
        except Exception as e:
            print('[baseline calibration] skipped:', e)
    baselines = build_baselines(n_true, denstream_kwargs=denstream_kw, dbstream_kwargs=dbstream_kw)
    dfs, extras = ({}, {})
    for name, model in {**variants, **baselines}.items():
        try:
            df, extra = run_stream_experiment(name, model, X_test, y_test, window=CONFIG['kdd']['window'], drift_log=drift_log_test)
            dfs[name] = df
            extras[name] = extra
        except Exception as e:
            print(f'  [{name}] FAILED: {e}')
            traceback.print_exc()
    print('[5/6] Statistical significance testing on the TEST segment ...')
    table7, table8 = build_significance_tables(dfs, reference='GTEC (full)', metric='ARI')
    table7.to_csv(os.path.join(TAB_DIR, 'Table7_statistical_significance.csv'), index=False)
    table8.to_csv(os.path.join(TAB_DIR, 'Table8_friedman_nemenyi_ranking.csv'), index=False)
    print('[6/6] Cross-dataset generalization check (Forest Covertype, frozen config, no re-tuning) ...')
    cov_table = pd.DataFrame()
    cov_results = None
    if CONFIG['covtype']['enabled']:
        Xc, yc, cov_desc, cov_class_names = load_covtype_stream(n_samples=CONFIG['covtype']['n_samples'], seed=RNG_SEED, block_reorder=CONFIG['covtype']['block_reorder'])
        if Xc is not None:
            n_true_cov = len(np.unique(yc))
            cov_model = make_gtec_factory(Xc.shape[1])()
            cov_baselines = build_baselines(n_true_cov)
            cov_dfs = {}
            for name, model in {'GTEC (full)': cov_model, **cov_baselines}.items():
                try:
                    df, _ = run_stream_experiment(name, model, Xc, yc, window=CONFIG['kdd']['window'], drift_log=None, verbose=False)
                    cov_dfs[name] = df
                except Exception as e:
                    print(f'  [covtype:{name}] FAILED: {e}')
            cov_rows = [{'method': name, 'ARI_mean': df['ARI'].mean(), 'NMI_mean': df['NMI'].mean(), 'Purity_mean': df['Purity'].mean()} for name, df in cov_dfs.items()]
            cov_table = pd.DataFrame(cov_rows).sort_values('ARI_mean', ascending=False) if cov_rows else cov_table
            cov_results = {'dfs': cov_dfs, 'X': Xc, 'y': yc, 'class_names': cov_class_names, 'desc': cov_desc}
    cov_table.to_csv(os.path.join(TAB_DIR, 'Table9_cross_dataset_generalization.csv'), index=False)
    results['models'] = {**variants, **baselines}
    results['dfs'] = dfs
    results['extras'] = extras
    results['drift_log'] = drift_log_test
    results['X'], results['y'] = (X_test, y_test)
    results['X_raw'] = Xraw_test
    results['class_names'] = class_names
    results['n_true'] = n_true
    results['kdd_desc'] = kdd_desc
    results['dataset_summary'] = dataset_summary
    results['sensitivity_df'] = sensitivity_df
    results['table7_significance'] = table7
    results['table8_friedman'] = table8
    results['table9_covtype'] = cov_table
    results['covtype_results'] = cov_results
    results['protocol'] = {'n_val': len(Xraw_val), 'n_test': len(Xraw_test), 'warmup_size': CONFIG['gtec']['warmup_size']}
    return results

def build_tables(results):
    dfs = results['dfs']
    rows = []
    for name, df in dfs.items():
        ex = results['extras'][name]
        valid_frac = float(df['ARI'].notna().mean()) if len(df) else 0.0
        rows.append({'method': name, 'ARI_mean': df['ARI'].mean(), 'ARI_std': df['ARI'].std(), 'NMI_mean': df['NMI'].mean(), 'NMI_std': df['NMI'].std(), 'Purity_mean': df['Purity'].mean(), 'Silhouette_mean': df['Silhouette'].mean(), 'n_valid_windows': int(df['ARI'].notna().sum()), 'n_total_windows': int(len(df)), 'valid_window_frac': round(valid_frac, 3), 'degenerate_run_flag': bool(valid_frac < 0.1), 'avg_ms_per_sample': ex['avg_ms_per_sample'], 'throughput_samples_per_s': ex['throughput_sps'], 'peak_memory_MB': ex['peak_memory_mb']})
    table2 = pd.DataFrame(rows).sort_values('ARI_mean', ascending=False)
    table2.to_csv(os.path.join(TAB_DIR, 'Table2_overall_performance.csv'), index=False)
    degenerate = table2[table2['degenerate_run_flag']]['method'].tolist()
    if degenerate:
        print(f'[WARNING] near-zero valid-window coverage (<10%), treat with caution: {degenerate}')
    min_len = min((len(df) for df in dfs.values()))
    common_valid = np.ones(min_len, dtype=bool)
    for df in dfs.values():
        common_valid &= df['ARI'].notna().values[:min_len]
    rows2b = []
    for name, df in dfs.items():
        sub = df.iloc[:min_len][common_valid]
        rows2b.append({'method': name, 'n_common_windows': int(common_valid.sum()), 'ARI_mean': sub['ARI'].mean(), 'NMI_mean': sub['NMI'].mean(), 'Purity_mean': sub['Purity'].mean()})
    table2b = pd.DataFrame(rows2b).sort_values('ARI_mean', ascending=False) if common_valid.sum() >= 5 else pd.DataFrame()
    table2b.to_csv(os.path.join(TAB_DIR, 'Table2b_common_windows_only.csv'), index=False)
    results['table2b_common_windows'] = table2b
    recov_frames = [ex['recovery_df'] for ex in results['extras'].values() if 'recovery_df' in ex]
    table3 = pd.concat(recov_frames, ignore_index=True) if recov_frames else pd.DataFrame()
    table3.to_csv(os.path.join(TAB_DIR, 'Table3_drift_recovery.csv'), index=False)
    abl_names = ['GTEC (full)', 'GTEC w/o History', 'GTEC w/o OT Memory (v3 nearest-centroid)', 'GTEC w/o Drift-Trigger', 'GTEC w/o Game (hard assign)', 'GTEC w/o Adaptive Utility']
    full_ari = results['dfs']['GTEC (full)']['ARI'].mean() if 'GTEC (full)' in results['dfs'] else np.nan
    abl_rows = []
    for name in abl_names:
        if name not in results['dfs']:
            continue
        df = results['dfs'][name]
        abl_rows.append({'variant': name, 'ARI_mean': df['ARI'].mean(), 'NMI_mean': df['NMI'].mean(), 'Purity_mean': df['Purity'].mean(), 'delta_ARI_vs_full': df['ARI'].mean() - full_ari})
    table4 = pd.DataFrame(abl_rows)
    table4.to_csv(os.path.join(TAB_DIR, 'Table4_ablation_study.csv'), index=False)
    table5 = build_complexity_table()
    table6 = results['sensitivity_df']
    table7 = results['table7_significance']
    table8 = results['table8_friedman']
    table9 = results['table9_covtype']
    print('\n' + '=' * 80)
    print('TABLE 2 — Overall performance comparison (KDD Cup 1999, TEST segment only)')
    print(table2.round(3).to_string(index=False))
    if len(table2b):
        print('\nTABLE 2b — Same comparison, common valid windows only (fair, apples-to-apples)')
        print(table2b.round(3).to_string(index=False))
    print('\nTABLE 4 — Ablation study (TEST segment)')
    print(table4.round(3).to_string(index=False))
    print('\nTABLE 6 — Sensitivity sweep (VALIDATION segment — diagnostic only, not tuned-to)')
    print(table6.round(3).to_string(index=False))
    if len(table7):
        print('\nTABLE 7 — Statistical significance vs. GTEC (full), paired Wilcoxon, Holm-corrected')
        print(table7.round(4).to_string(index=False))
    if len(table8):
        print('\nTABLE 8 — Friedman/Nemenyi average-rank ranking (lower rank = better)')
        print(table8.round(3).to_string(index=False))
    if len(table9):
        print('\nTABLE 9 — Cross-dataset generalization (Forest Covertype, frozen config)')
        print(table9.round(3).to_string(index=False))
    print('=' * 80)
    return {'Table1': results['dataset_summary'], 'Table2': table2, 'Table2b': table2b, 'Table3': table3, 'Table4': table4, 'Table5': table5, 'Table6': table6, 'Table7': table7, 'Table8': table8, 'Table9': table9}

def savefig(fig, fname):
    path = os.path.join(FIG_DIR, fname)
    fig.tight_layout()
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    print(f'  saved {fname}')

def fig1_drift_monitor(results):
    model = results['models'].get('GTEC (full)')
    if model is None:
        return
    fig, ax = plt.subplots(figsize=(10, 4))
    if model.drift_events:
        for i, dt in enumerate(model.drift_events):
            ax.axvline(dt, color='crimson', alpha=0.6, lw=1, label='GTEC-detected Drift-Equilibrium event' if i == 0 else None)
    for i, e in enumerate(results['drift_log']):
        ax.axvline(e['t'], color='steelblue', ls='--', alpha=0.5, label='real ground-truth label-shift event' if i == 0 else None)
    ax.set_title('Fig. 1 — Drift-Equilibrium monitor on the real KDD stream:\nground-truth attack-category shifts vs. GTEC-detected structural events')
    ax.set_xlabel('stream time t')
    ax.set_yticks([])
    ax.legend(loc='upper right', fontsize=8)
    savefig(fig, 'Fig1_drift_equilibrium_monitor.png')

def fig2_quality_over_time(results):
    dfs = results['dfs']
    main_methods = ['GTEC (full)', 'CluStream', 'DenStream', 'DBSTREAM', 'StreamKM++']
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for name in main_methods:
        if name not in dfs:
            continue
        df = dfs[name]
        axes[0].plot(df['t'], df['ARI'], marker='o', ms=3, label=name)
        axes[1].plot(df['t'], df['NMI'], marker='o', ms=3, label=name)
    for e in results['drift_log']:
        axes[0].axvline(e['t'], color='grey', ls=':', alpha=0.35)
        axes[1].axvline(e['t'], color='grey', ls=':', alpha=0.35)
    axes[0].set_title('ARI over time — KDD Cup 1999 stream')
    axes[1].set_title('NMI over time — KDD Cup 1999 stream')
    for ax in axes:
        ax.set_xlabel('stream time t')
        ax.set_ylim(-0.05, 1.05)
    axes[0].set_ylabel('Adjusted Rand Index')
    axes[1].set_ylabel('Normalized Mutual Info')
    axes[1].legend(fontsize=8, loc='lower left')
    savefig(fig, 'Fig2_quality_over_time_kdd.png')

def fig3_recovery_by_type(table3):
    if table3 is None or len(table3) == 0:
        return
    piv = table3.pivot_table(index='drift_type', columns='method', values='recovery_time_samples', aggfunc='mean')
    fig, ax = plt.subplots(figsize=(10, 4.8))
    piv.plot(kind='bar', ax=ax)
    ax.set_title('Fig. 3 — Equilibrium Recovery Time by real attack-category transition\n(lower = better)')
    ax.set_ylabel('samples until quality recovers to 90% of pre-drift ARI')
    ax.set_xlabel('ground-truth transition (into attack class)')
    ax.legend(fontsize=7, ncol=2)
    savefig(fig, 'Fig3_recovery_time_by_transition.png')

def fig4_runtime_memory(table2):
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    ax.scatter(table2['avg_ms_per_sample'], table2['ARI_mean'], s=table2['peak_memory_MB'].clip(lower=1) * 1.2, alpha=0.7, c='teal')
    for _, r in table2.iterrows():
        ax.annotate(r['method'], (r['avg_ms_per_sample'], r['ARI_mean']), fontsize=8, xytext=(4, 4), textcoords='offset points')
    ax.set_xlabel('avg processing time per sample (ms)')
    ax.set_ylabel('mean ARI')
    ax.set_title('Fig. 4 — Quality vs. latency vs. memory footprint (bubble size), KDD stream')
    savefig(fig, 'Fig4_runtime_memory_tradeoff.png')

def fig5_ablation(table4):
    if table4 is None or len(table4) == 0:
        return
    fig, ax = plt.subplots(figsize=(8, 4.8))
    x = np.arange(len(table4))
    ax.bar(x, table4['ARI_mean'], color='teal', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(table4['variant'], rotation=25, ha='right', fontsize=8)
    ax.set_ylabel('mean ARI (KDD stream)')
    ax.set_title('Fig. 5 — Ablation study: contribution of each GTEC component')
    for xi, v in zip(x, table4['ARI_mean']):
        ax.text(xi, v + 0.01, f'{v:.2f}', ha='center', fontsize=8)
    savefig(fig, 'Fig5_ablation_study.png')

def fig6_pca_snapshot(results):
    model = results['models'].get('GTEC (full)')
    if model is None or not results['drift_log']:
        return
    X, y = (results['X'], results['y'])
    event = results['drift_log'][len(results['drift_log']) // 2]
    t_evt = event['t']
    before_idx = max(0, t_evt - 400)
    after_idx = min(len(X) - 1, t_evt + 400)
    viz_pca = PCA(n_components=2, random_state=RNG_SEED).fit(X)
    proj = viz_pca.transform(X)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, idx, label in [(axes[0], before_idx, 'before'), (axes[1], after_idx, 'after')]:
        window = slice(max(0, idx - 200), idx + 200)
        ax.scatter(proj[window, 0], proj[window, 1], c=y[window], cmap='tab10', s=12, alpha=0.8)
        ax.set_title(f"t≈{idx} ({label} '{event['detail']}' @ t={t_evt})")
    fig.suptitle(f"Fig. 6 — 2-D PCA snapshot of the real KDD stream, true attack classes,\nbefore/after a real '{event['detail']}' concept-drift event")
    savefig(fig, 'Fig6_pca_before_after_drift.png')

def fig7_clusters_over_time(results):
    fig, ax = plt.subplots(figsize=(9.5, 4.8))
    main_methods = ['GTEC (full)', 'CluStream', 'DenStream', 'DBSTREAM', 'StreamKM++']
    for name in main_methods:
        if name not in results['dfs']:
            continue
        df = results['dfs'][name]
        ax.plot(df['t'], df['n_clusters'], marker='o', ms=3, label=name)
    for e in results['drift_log']:
        ax.axvline(e['t'], color='grey', ls=':', alpha=0.3)
    ax.set_title('Fig. 7 — Active cluster count over time (structural adaptivity:\nbirth / split / merge / disappear dynamics), KDD stream')
    ax.set_xlabel('stream time t')
    ax.set_ylabel('# active clusters / micro-clusters')
    ax.legend(fontsize=8)
    savefig(fig, 'Fig7_active_clusters_over_time.png')

def fig8_purity_heatmap(results):
    model = results['models'].get('GTEC (full)')
    if model is None:
        return
    X, y = (results['X'], results['y'])
    class_names = results['class_names']
    preds = np.array([model.predict_one(xx) for xx in X])
    valid = preds >= 0
    if valid.sum() < 2:
        return
    pred_ids = sorted(np.unique(preds[valid]).tolist())
    mat = np.zeros((len(pred_ids), len(class_names)))
    for i, pid in enumerate(pred_ids):
        mask = (preds == pid) & valid
        for j in range(len(class_names)):
            mat[i, j] = np.sum((y == j) & mask)
    row_sums = mat.sum(axis=1, keepdims=True)
    mat_norm = np.divide(mat, np.maximum(row_sums, 1))
    fig_h = max(3.5, 0.45 * len(pred_ids) + 2)
    fig_w = max(6.0, 1.3 * len(class_names) + 3)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    im = ax.imshow(mat_norm, aspect='auto', cmap='YlGnBu', vmin=0, vmax=1)
    ax.set_xticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=30, ha='right')
    ax.set_yticks(range(len(pred_ids)))
    ax.set_yticklabels([f'cluster {p}' for p in pred_ids], fontsize=8)
    for i in range(len(pred_ids)):
        for j in range(len(class_names)):
            if mat_norm[i, j] > 0.01:
                ax.text(j, i, f'{mat_norm[i, j]:.2f}', ha='center', va='center', color='white' if mat_norm[i, j] > 0.5 else 'black', fontsize=7)
    ax.set_title('Fig. 8 — GTEC cluster-purity composition vs. true attack classes\n(row-normalized; full KDD stream, final cluster state)')
    fig.colorbar(im, ax=ax, shrink=0.8, label="fraction of cluster's points")
    savefig(fig, 'Fig8_cluster_purity_heatmap.png')

def fig9_sensitivity(sens_df):
    if sens_df is None or len(sens_df) == 0:
        return
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(sens_df['value'], sens_df['ARI_mean'], marker='o', label='mean ARI')
    ax.plot(sens_df['value'], sens_df['NMI_mean'], marker='s', label='mean NMI')
    ax.set_xlabel(str(sens_df['param'].iloc[0]))
    ax.set_ylabel('mean metric value')
    ax.set_title('Fig. 9 — Sensitivity of GTEC to the self-tuning\nDrift-Equilibrium threshold (k_sigma) — VALIDATION segment')
    ax.legend()
    savefig(fig, 'Fig9_hyperparameter_sensitivity.png')

def fig10_ot_transport_plan(results):
    model = results['models'].get('GTEC (full)')
    if model is None or getattr(model, 'last_ot_plan', None) is None:
        print('  [Fig10] skipped — no OT-memory reuse event was triggered on the TEST segment.')
        return
    info = model.last_ot_plan
    plan = info['plan']
    fig, ax = plt.subplots(figsize=(max(5.0, 0.55 * plan.shape[1] + 3), max(4.0, 0.4 * plan.shape[0] + 2)))
    im = ax.imshow(plan, aspect='auto', cmap='magma')
    ax.set_xlabel('historical prototype (retrieved stable regime)')
    ax.set_ylabel('current micro-cluster')
    ax.set_xticks(range(plan.shape[1]))
    ax.set_yticks(range(plan.shape[0]))
    ax.set_title(f'Fig. 10 — Optimal-Transport historical-memory match\n(barycentric blend weights; reuse event @ t={info['t']}, TEST segment)')
    fig.colorbar(im, ax=ax, shrink=0.85, label='transport mass')
    savefig(fig, 'Fig10_ot_memory_transport_plan.png')

def fig11_protocol_timeline(results):
    proto = results.get('protocol', {})
    n_val, n_test, warm = (proto.get('n_val'), proto.get('n_test'), proto.get('warmup_size'))
    if not n_val or not n_test:
        return
    fig, ax = plt.subplots(figsize=(10, 2.6))
    ax.barh(0, n_val, color='#4C72B0', label='VALIDATION — tuning only (PCA fit, sensitivity sweep)')
    ax.barh(0, n_test, left=n_val, color='#55A868', label='TEST — touched once (Tables 2/3/4/7/8, Figs. 1-8/10)')
    if warm:
        ax.barh(0, min(warm, n_test), left=n_val, color='#C44E52', alpha=0.9, label="TEST-segment warm-start (each model's own fresh offline init)")
    ax.set_yticks([])
    ax.set_xlabel('stream index (chronological — no shuffling)')
    ax.set_title('Fig. 11 — Validation / test protocol: what each segment is allowed to see')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.4), ncol=1, fontsize=8, frameon=False)
    savefig(fig, 'Fig11_val_test_protocol_timeline.png')

def fig12_critical_difference(table8):
    if table8 is None or len(table8) == 0:
        return
    df = table8.sort_values('avg_rank').reset_index(drop=True)
    cd = float(df['critical_difference'].iloc[0])
    y_pos = np.arange(len(df))[::-1]
    fig, ax = plt.subplots(figsize=(8.5, 0.6 * len(df) + 1.8))
    ax.scatter(df['avg_rank'], y_pos, s=70, color='teal', zorder=3)
    for yi, m, r in zip(y_pos, df['method'], df['avg_rank']):
        ax.text(r, yi + 0.18, f'{m}  (rank {r:.2f})', ha='center', fontsize=7.5)
    best_rank = float(df['avg_rank'].min())
    ax.axvspan(best_rank, best_rank + cd, color='teal', alpha=0.12, label=f'critical difference = {cd:.2f} (alpha=0.05)')
    ax.set_yticks([])
    ax.set_xlabel('average rank across TEST-segment windows (1 = best)')
    ax.set_title('Fig. 12 — Friedman/Nemenyi critical-difference ranking\n(methods inside the shaded band of the top rank are not significantly different)')
    ax.legend(fontsize=8, loc='lower right')
    savefig(fig, 'Fig12_critical_difference_diagram.png')

def fig13_cross_dataset(table9):
    if table9 is None or len(table9) == 0:
        print('  [Fig13] skipped — cross-dataset generalization check produced no results (Forest Covertype unavailable, e.g. no internet access).')
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    x = np.arange(len(table9))
    for ax, metric in zip(axes, ['ARI_mean', 'NMI_mean']):
        colors = ['crimson' if m == 'GTEC (full)' else 'steelblue' for m in table9['method']]
        ax.bar(x, table9[metric], color=colors, alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(table9['method'], rotation=30, ha='right', fontsize=8)
        ax.set_ylabel(metric.replace('_mean', ''))
    fig.suptitle('Fig. 13 — Cross-dataset generalization: Forest Covertype\n(frozen CONFIG from the KDD validation step — no re-tuning)')
    savefig(fig, 'Fig13_cross_dataset_generalization.png')

def build_figures(results, tables):
    print('\nGenerating figures ...')
    fig1_drift_monitor(results)
    fig2_quality_over_time(results)
    fig3_recovery_by_type(tables['Table3'])
    fig4_runtime_memory(tables['Table2'])
    fig5_ablation(tables['Table4'])
    fig6_pca_snapshot(results)
    fig7_clusters_over_time(results)
    fig8_purity_heatmap(results)
    fig9_sensitivity(tables['Table6'])
    fig10_ot_transport_plan(results)
    fig11_protocol_timeline(results)
    fig12_critical_difference(tables['Table8'])
    fig13_cross_dataset(tables['Table9'])
README_TEMPLATE = '# GTEC v4 — Experiment Outputs (real KDD Cup 1999 + Forest Covertype data only)\n\nGenerated automatically by `code.py`. Re-running the script reproduces\nidentical numbers (see "Determinism" below).\n\n## Method\nGTEC (Game-theoretic Equilibrium Clustering) implements: a local\nevolutionary game over existing candidate clusters PLUS an explicit "found\na new cluster" strategy, resolved by multi-iteration replicator dynamics;\nan adaptive multi-term utility function (similarity, density, temporal\nstability, historical compatibility, dispersion, memory cost, structural\nage risk, plus a dedicated novelty term for the birth strategy); a\nself-tuning Drift-Equilibrium indicator D_E(t); triggered Split/Merge/\nMaintain/Disappear cluster evolution; and a bounded Historical Equilibrium\nMemory, built from past STABLE regimes, matched via entropic-regularized\nOptimal Transport and reused via an OT barycentric blend (v4) instead of a\nsingle-nearest-centroid rule.\n\n## What changed, and why (full version history)\n`code.py`\'s module docstring has the complete changelog; read it before\nciting any number from this run. Summary:\n- Density term scaled by similarity; History-blend confidence gating\n  has no floor (eff_alpha = alpha * conf, both OT and v3 paths).\n- Root eval fix kept: a single-cluster window scores a real ARI of 0,\n  not a discarded NaN. window=100 for paired-test power.\n- River baseline predict_one falls back to nearest known cluster on\n  a -1 return (not just a raised exception); DenStream/DBSTREAM are\n  also calibrated on VALIDATION only before the TEST run.\n- StandardScaler fit on VALIDATION only, applied to TEST (was\n  leaking TEST statistics into normalization before). Covertype\n  scaler fit on a causal leading prefix, not the whole sequence.\n- min_active_clusters floor added to merge/disappear, plus a capped\n  birth-utility scale, fixing a confirmed single-cluster lock-in\n  mode where a large merge inflated a cluster radius enough that\n  birth utility could never recover for the rest of the stream.\n- Table 2 reports valid_window_frac and flags degenerate runs.\n- Table 8 uses a greedy joint-overlap method selection instead of a\n  flat per-method coverage threshold.\n- Table 2b (fair, common-valid-window-only comparison) kept.\n- Drift recovery requires 2 consecutive qualifying windows, not 1.\n- Library versions logged at startup for reproducibility.\n- **v1 -> v2**: fixed a curse-of-dimensionality bug in the density term and\n  an under-estimated cluster radius; replaced fixed manual thresholds with\n  self-tuning statistics; added an unsupervised warm-start phase; added a\n  shared causal PCA stage; switched to ground-truth drift events mined\n  from KDD\'s own real label sequence.\n- **v2 -> v3**: reverted a soft multi-cluster statistical blend (measured\n  as harmful on real data) in favor of a hard update chosen by an\n  evolutionary game that includes cluster-birth as a competing strategy;\n  fixed Historical Memory to snapshot STABLE regimes only; removed a\n  duplicated novelty term.\n- **v3 -> v4**: (1) historical-memory matching/reuse now goes through\n  mass-weighted Optimal Transport with an OT-barycentric blend, kept\n  ablatable against the exact v3 rule; (2) every reported number now comes\n  from a single pass over a held-out TEST segment, with all tuning\n  (the sensitivity sweep, PCA fitting) restricted to a chronologically\n  earlier VALIDATION segment it never sees; (3) paired Wilcoxon\n  (Holm-corrected) and Friedman/Nemenyi significance testing were added\n  instead of reporting point estimates alone; (4) a second, independent\n  real dataset (Forest Covertype) is evaluated with the exact config\n  frozen from KDD validation, as a no-retuning generalization check.\n\n## Datasets\n- **KDD Cup 1999 (10%)** — real-world network-intrusion stream, chosen for\n  direct comparability with the CluStream/DenStream literature. Ground-truth\n  concept-drift events are mined from the real coarse-label sequence and\n  used ONLY for evaluation, never fed to any model. (If no internet access\n  was available at run time, a small synthetic fallback stream was used\n  instead — check the printed log / Table 1 to see which happened.)\n- **Forest Covertype (UCI/sklearn)** — second, independent real dataset,\n  used only for the frozen-config generalization check (Table 9 / Fig. 13).\n  Not a native stream: a disclosed, seeded class-block reordering is\n  applied to give it an evaluable drift-like arrival order (see\n  `load_covtype_stream()`\'s docstring) — this is explicitly NOT presented\n  as organic drift the way the KDD attack sequence is.\n\n## Validation / test protocol\nSee `CONFIG["protocol"]` and the v3->v4 changelog entry above, and Fig. 11\nfor a visual timeline. In one sentence: the sensitivity sweep and PCA\nfitting only ever see the VALIDATION segment; Tables 2/3/4/7/8 and most\nfigures come from one untouched pass over the TEST segment with freshly\nre-initialized models.\n\n## Baselines\nCluStream, DenStream, DBSTREAM and a StreamKM++-style online k-means,\npreferentially instantiated from the `river` streaming ML library, with\ncompact from-scratch fallbacks if `river` was unavailable.\n\n## Statistical testing\nTable 7: per-window ARI paired between GTEC (full) and every other method\non the SAME test-segment windows, Wilcoxon signed-rank test, Holm-Bonferroni\ncorrected across the comparison family. Table 8: Friedman test (blocks =\nwindows, treatments = methods) with Nemenyi critical-difference average\nranks (Fig. 12). Comparisons that aren\'t significant at alpha=0.05 are\nreported as such, not omitted.\n\n## Determinism\nEvery RNG entry point (KMeans, PCA, silhouette subsampling, the model\nconstructors) is seeded from a single `RNG_SEED = 42`, and every BLAS\nthread pool (OMP/OpenBLAS/MKL) is pinned to 1 thread before numpy is even\nimported. Re-running `python code.py` on the same machine/environment\nshould reproduce identical tables and figures. (Results can still differ\nacross different machines/OS/library versions due to differing BLAS\nimplementations — this is a property of floating-point arithmetic, not of\nthis script\'s seeding.)\n\n## Contents (13 figures, 9 tables)\n- `figures/` — Figs. 1-9 as in v3 (now computed on the TEST segment only),\n  plus **Fig. 10** OT historical-memory transport plan, **Fig. 11**\n  validation/test protocol timeline, **Fig. 12** Friedman/Nemenyi\n  critical-difference diagram, **Fig. 13** cross-dataset (Covertype)\n  generalization bars.\n- `tables/` — Tables 1-6 as in v3 (Table 6 relabeled VALIDATION-only),\n  plus **Table 7** paired statistical significance, **Table 8**\n  Friedman/Nemenyi ranking, **Table 9** cross-dataset generalization.\n- `raw_results/` — full per-window metric logs (CSV) for every method on\n  both datasets, plus a `run_config.json` snapshot of all hyperparameters\n  (the exact, frozen config both datasets were evaluated with).\n- `code.py` — this script, for full reproducibility.\n\n## Re-running / scaling up\nEdit `CONFIG` at the top of `code.py` (`n_samples`, `window`, GTEC\nhyperparameters, `protocol.val_frac`) and re-run. Defaults are kept modest\nfor a fast first run; increase sample size for final paper-quality numbers.\nChanging any GTEC hyperparameter based on a TEST-segment number defeats the\npoint of the v4 protocol — re-tune using the VALIDATION segment\n(`run_sensitivity_sweep` / Table 6) instead.\n'

def package_outputs(results, tables):
    for name, df in results['dfs'].items():
        safe = name.replace(' ', '_').replace('/', '-').replace('(', '').replace(')', '')
        df.to_csv(os.path.join(RAW_DIR, f'kdd_test__{safe}.csv'), index=False)
    cov = results.get('covtype_results')
    if cov is not None:
        for name, df in cov['dfs'].items():
            safe = name.replace(' ', '_').replace('/', '-').replace('(', '').replace(')', '')
            df.to_csv(os.path.join(RAW_DIR, f'covtype__{safe}.csv'), index=False)
    with open(os.path.join(RAW_DIR, 'run_config.json'), 'w') as f:
        json.dump(CONFIG, f, indent=2, default=str)
    with open(os.path.join(OUT_DIR, 'README.md'), 'w') as f:
        f.write(README_TEMPLATE)
    try:
        shutil.copy(os.path.abspath(__file__), os.path.join(OUT_DIR, 'code.py'))
    except Exception:
        pass
    zip_path = os.path.join(os.path.dirname(OUT_DIR) or '.', 'GTEC_Q1_Outputs.zip')
    if os.path.exists(zip_path):
        os.remove(zip_path)
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(OUT_DIR):
            for fn in sorted(files):
                full = os.path.join(root, fn)
                arcname = os.path.relpath(full, OUT_DIR)
                zf.write(full, arcname)
    print(f'\nAll outputs zipped to: {zip_path}')
    try:
        from google.colab import files as colab_files
        colab_files.download(zip_path)
    except Exception:
        print(f'(Not running in Colab, or auto-download unavailable — retrieve the zip manually from {zip_path})')
    return zip_path
if __name__ == '__main__':
    t_start = time.time()
    results = run_all()
    tables = build_tables(results)
    build_figures(results, tables)
    zip_path = package_outputs(results, tables)
    print(f'\nDone in {time.time() - t_start:.1f}s total.')
    print(f'Download / find your results at: {zip_path}')

GTEC v4 Q1 ARTICLE EXPERIMENT PIPELINE — real KDD Cup 1999 + Forest Covertype streams
river library available: True
Determinism: RNG_SEED=42, BLAS threads pinned to 1
Library versions: numpy=2.1.3 pandas=2.2.3 sklearn=1.6.1 scipy=1.16.3 river=0.26.1
Output directory: /content/gtec_outputs

[1/6] Loading the real-world KDD Cup 1999 (10%) stream ...


Exception ignored on calling ctypes callback function <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7aff6a4a3b00>:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/usr/local/lib/python3.13/dist-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
  File "/usr/local/lib/python3.13/dist-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
  File "/usr/lib/python3.13/ctypes/__init__.py", line 361, in __init__
    self._handle = self._load_library(name, mode, handle, winmode)
  File "/usr/lib/python3.13/ctypes/__init__.py", line 403, in _load_library
    return _dlopen(name, mode)
OSError: /usr/local/lib/python3.13/dist-packages/numpy.libs/libscipy_openblas64_-ff651d7f.so: cannot 

[data] KDD Cup 1999 (10%) loaded: 16000 samples, 41 raw features, 5 coarse attack classes (dos, normal, probe, r2l, u2r).
[protocol] 5600 samples -> VALIDATION (tuning only), 10400 samples -> TEST (touched once, for every reported number below).
[data] PCA representation: 41 -> 13 dims (95.6% variance retained, fit on the first 1200 samples as a causal 'reference phase').
[data] 5 real concept-drift (label-shift) events detected in the TEST segment's ground-truth sequence (used only for evaluation).

[2/6] Hyperparameter sensitivity sweep — VALIDATION segment only ...
  [sensitivity] drift_k_sigma=1.5: mean ARI=0.678, mean NMI=0.684
  [sensitivity] drift_k_sigma=3.0: mean ARI=0.658, mean NMI=0.669
  [sensitivity] drift_k_sigma=4.5: mean ARI=0.658, mean NMI=0.669
[3/6] Running proposed method (GTEC) + ablations on the TEST segment ...
[4/6] Running baselines (CluStream, DenStream, DBSTREAM, StreamKM++) on the TEST segment ...
[baseline] river DenStream unavailable, using fallback: math 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done in 185.3s total.
Download / find your results at: /content/GTEC_Q1_Outputs.zip
